# LLM APIs & Chat Completions

*Day 1 of Week 1.* Everything we build over the next 4 weeks — retrieval-augmented generation (RAG), agents, evaluation pipelines, and production deployment — sits on top of one primitive: a call to an **LLM API**. Before reaching for frameworks, we want a solid understanding of that primitive itself.

In this notebook we cover **tokens and cost**, basic **completions**, **streaming**, **structured output** via Pydantic, and **tool use** (or function calling). We close by assembling a small `LLMClient` wrapper class that every subsequent notebook in this series imports. Understanding the API at this level lets you discuss cost trade-offs and system architecture confidently in interviews.

## What Is an LLM API?

A **large language model** (LLM) is, at its core, a learned probability distribution over the next token given all preceding tokens:

$$p(x_t \mid x_{<t}; \Theta).$$

At inference time, the model generates text **autoregressively**: it samples one token from this distribution, appends it to the sequence, and repeats until it emits a special stop token or reaches `max_tokens`. The quality and diversity of the output are controlled by **temperature**[^temp] $0 \leq T < \infty$. At $T = 0$ the model always picks the highest-probability token (greedy decoding); at $T > 1$ the distribution starts to flatten and outputs become more varied and sometimes incoherent.

[^temp]: Temperature divides logits prior to softmax: $\textbf{\textsf{p}} = \text{Softmax}(\textbf{\textsf{z}} / T).$ So $T = 0$ is essentially argmax on the largest logit, while $T > 1$ smooths out the logits: $\exp(\textbf{\textsf{z}} / T) = \exp(\textbf{\textsf{z}})^{1/T}$.

**Chat protocol.** Modern APIs expose a structured conversation format with three message roles: `system` (persistent instructions setting the model's persona and task), `user` (the human turn), and `assistant` (prior model turns). Internally the model sees all of these concatenated into a single prompt — the conversation history is just a prefix that conditions the next token. [There is no server-side memory; every API call is stateless]{.underline}. If you want the model to remember prior turns, you must resend them yourself.

**Key parameters.** Beyond `temperature`, the two most common sampling controls are `top_p` ([nucleus sampling](https://arxiv.org/abs/1904.09751): restrict sampling to the smallest set of tokens whose cumulative probability exceeds $p$) and `max_tokens` (a hard cap on output length). The `stop` parameter accepts a list of strings at which generation halts. In practice, for production financial AI systems we set `temperature=0` for reproducibility and auditability — the same prompt should produce the same answer every time.

**What the API does not do.** The model has no persistent memory across calls, no internet access by default, and no awareness of events after its training cutoff. It cannot look up live stock prices, query your database, or recall a conversation from yesterday — unless you explicitly give it those tools, which we cover in the Tool Use section.

**Model tiers.** Different models trade quality for cost: `gpt-4o` is the highest-capability OpenAI model; `gpt-4o-mini` is a smaller, faster, and dramatically cheaper variant. Choosing the right model per pipeline stage is one of the most important cost-engineering decisions we make. We quantify this in the next section.

## Tokens and Cost

LLMs do not operate on characters or words — they operate on **tokens**, the output of a learned subword tokenizer called **Byte-Pair Encoding** (BPE). Common English words are usually a single token; rare, technical, or compound words are split into multiple subword pieces. As a rule of thumb, $\approx 0.75$ English words correspond to one token.

Pricing is per token, billed separately for input (prompt) and output (completion) tokens. We pay more for output tokens because each one requires a full forward pass. The cost model is:

$$\text{cost} = \frac{n_{\text{input}} \cdot p_{\text{input}} + n_{\text{completion}} \cdot p_{\text{output}}}{10^6}$$

where $p_{\text{input}}$ and $p_{\text{output}}$ are prices per million tokens.

In [ ]:
#| echo: false
import os
from dotenv import load_dotenv
load_dotenv()

We use the `tiktoken` library — OpenAI's tokenizer — to count tokens without making any API calls:

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

samples = [
    "Hello!",
    "The Federal Reserve raised interest rates by 25 basis points.",
    "Financialization of the economy refers to the increasing role of financial motives, financial markets, financial actors and financial institutions in the operation of domestic and international economies.",
]
for t in samples:
    n = count_tokens(t)
    print(f"{n:4d} tokens | {t[:70]}")

Note how "Financialization" is split into multiple subword tokens — BPE handles rare and technical vocabulary by breaking it into common subword pieces. Domain-specific financial jargon tends to tokenize less efficiently than plain English prose.

We write an `estimate_cost` function to reason about API spend at design time:

In [ ]:
PRICES = {                                  # USD per 1M tokens
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

def estimate_cost(
    prompt: str,
    completion: str,
    model: str = "gpt-4o-mini",
) -> float:
    """Estimate cost in USD for one API call."""
    n_prompt = count_tokens(prompt, model)
    n_completion = count_tokens(completion, model)
    p = PRICES[model]
    return (n_prompt * p["input"] + n_completion * p["output"]) / 1_000_000

# Simulate a RAG query: long prompt, short answer
prompt = (
    "You are a financial analyst. Answer based on the following context.\n\n"
    "Context: " + ("The company reported strong earnings growth. " * 50) + "\n\n"
    "Question: What did the company report?"
)
completion = "The company reported strong earnings growth."

for model in PRICES:
    cost = estimate_cost(prompt, completion, model)
    print(f"{model:20s}: ${cost:.6f}")

The cost difference is roughly $17\times$. For a RAG pipeline processing 10,000 queries per day, that gap determines whether a feature is economically viable. We choose models deliberately per stage: `gpt-4o-mini` for retrieval grading and cheap classification calls, `gpt-4o` for final synthesis of complex multi-document questions where quality matters most.

## First Completions

The OpenAI client reads `OPENAI_API_KEY` from the environment automatically. The response object contains the generated text, token usage counts, and a `finish_reason` that tells us why generation stopped.

In [ ]:
import openai

client = openai.OpenAI()  # reads OPENAI_API_KEY from env # <1>

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is the Fed funds rate?"}],
    temperature=0.0,
    max_tokens=200,
)

print(response.choices[0].message.content)         # <2>
print(f"\nPrompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Finish reason:     {response.choices[0].finish_reason}")  # <3>

1. The client is initialized once and reused across calls. It picks up `OPENAI_API_KEY` automatically from the environment.
2. `choices[0]` because the API can return multiple completions when `n > 1`; we always default to one.
3. `finish_reason` values: `"stop"` (normal end of generation), `"length"` (hit `max_tokens` limit), `"tool_calls"` (model wants to invoke a function).

We run a small temperature ablation to see how the sampling sharpness parameter affects output:

In [ ]:
#| code-fold: true
question = "In one sentence, describe the main risk of holding long-duration bonds."
for temp in [0.0, 0.7, 1.5]:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=temp,
        max_tokens=80,
    )
    print(f"T={temp}: {r.choices[0].message.content.strip()}\n")

At $T=0$ the output is deterministic — the same prompt produces the same answer on every call. For production financial AI systems we default to $T=0$ to ensure reproducible, auditable responses.

## Streaming

For interactive applications, streaming surfaces the first token immediately rather than waiting for the entire response to be generated server-side. This is the difference between a user seeing text appear in under a second vs. waiting 5+ seconds for a long answer. The `stream=True` parameter returns a generator of `ChatCompletionChunk` objects, each carrying a `delta` — the incremental new text.

In [ ]:
import time

def stream_completion(
    messages: list[dict],
    model: str = "gpt-4o-mini",
) -> str:
    """Stream completion tokens to stdout, return full text."""
    start = time.time()
    full_text = ""
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        max_tokens=200,
        temperature=0.0,
    )
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""  # <1>
        print(delta, end="", flush=True)
        full_text += delta
    elapsed = time.time() - start
    print(f"\n\n[{elapsed:.2f}s total]")
    return full_text

_ = stream_completion([
    {"role": "user", "content": "Explain duration risk in bonds in two sentences."}
])

1. Each chunk contains a `delta` — the incremental new text. Early chunks and the final chunk may carry an empty `delta`, so we default to `""` rather than `None`.

## Structured Output with Pydantic

Raw text output is fragile in production — models may wrap JSON in markdown code fences, omit required fields, or produce values of the wrong type. The `response_format` parameter with a Pydantic model gives us validated, typed output enforced at the API level. We define the schema as a `BaseModel` subclass and pass it directly.

In [ ]:
from pydantic import BaseModel
from typing import Optional

class FinancialEntity(BaseModel):
    company_name: str
    ticker: Optional[str] = None
    metric: str
    value: float
    unit: str
    fiscal_period: str

text = "Apple reported revenue of $94.9 billion for Q1 FY2024."

response = client.beta.chat.completions.parse(  # <1>
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract the financial entity from the text."},
        {"role": "user", "content": text},
    ],
    response_format=FinancialEntity,
)
entity: FinancialEntity = response.choices[0].message.parsed  # <2>
print(entity.model_dump_json(indent=2))

1. `client.beta.chat.completions.parse` is the recommended method for Pydantic structured output — it validates the JSON against the schema server-side and raises a `ValidationError` if the model returns something that does not fit.
2. `message.parsed` returns the already-validated Pydantic object, not raw JSON.

:::{.callout-caution}
Use `response_format` for all production data extraction. The alternative — parsing JSON from the model's free-text response — breaks on edge cases like nested quotes, trailing commas, or markdown code fences wrapping the output.

:::

## Tool Use (Function Calling)

Tool use is the primitive underlying all agentic systems. LLMs know *what* to do but they have no knowledge of the current state of the world — stock prices, database records, today's date, or any data produced after their training cutoff. The API bridges this gap with a two-turn protocol: in the first call the model inspects the available tool schemas and returns a `tool_calls` object (not text); we execute the referenced function locally; in the second call the model receives the result and synthesizes a final natural-language answer.

We define a mock stock price tool and its JSON schema:

In [ ]:
import json

def get_stock_price(ticker: str) -> dict:
    """Return a mock current stock price."""
    prices = {"AAPL": 189.50, "MSFT": 415.20, "GS": 492.30, "JPM": 198.75}
    return {
        "ticker": ticker,
        "price": prices.get(ticker, 100.0),
        "currency": "USD",
    }

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price for a ticker symbol.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "Stock ticker symbol, e.g. AAPL",
                    }
                },
                "required": ["ticker"],
            },
        },
    }
]

We implement the dispatch loop that runs until the model produces a final text response:

In [ ]:
def run_tool_loop(
    messages: list[dict],
    tools: list[dict],
    tool_registry: dict,
) -> str:
    """Run tool calls until the model produces a final text response."""
    messages = list(messages)  # don't mutate the caller's list
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            temperature=0.0,
        )
        msg = response.choices[0].message
        if response.choices[0].finish_reason != "tool_calls":  # <1>
            return msg.content
        messages.append(msg)                                    # <2>
        for tc in msg.tool_calls:
            fn = tool_registry[tc.function.name]
            args = json.loads(tc.function.arguments)
            result = fn(**args)
            messages.append({                                   # <3>
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })

answer = run_tool_loop(
    messages=[{"role": "user", "content": "What is the current price of AAPL?"}],
    tools=TOOLS,
    tool_registry={"get_stock_price": get_stock_price},
)
print(answer)

1. Loop until `finish_reason` is not `"tool_calls"` — meaning the model has synthesized a final natural-language response.
2. Append the assistant's tool-call message to maintain the full conversation state before the next turn.
3. Each tool result is appended as a `"tool"` role message keyed by `tool_call_id`, so the model knows which call this result answers.

## The `LLMClient` Wrapper

We consolidate everything into a reusable `LLMClient` class. Every notebook in this series imports this class — it provides a consistent interface for plain completions, structured output, streaming, and tool loops, and it accumulates cost across all calls so we can measure the true API spend of any pipeline we build.

In [ ]:
from typing import Callable, Generator, Type
from pydantic import BaseModel
import openai, json

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}


class LLMClient:
    """Thin wrapper around the OpenAI chat API with cost tracking."""

    def __init__(self, model: str = "gpt-4o-mini", temperature: float = 0.0):
        self.model = model
        self.temperature = temperature
        self._client = openai.OpenAI()
        self._total_input_tokens = 0
        self._total_output_tokens = 0

    # ------------------------------------------------------------------
    # Public interface
    # ------------------------------------------------------------------

    def complete(
        self,
        messages: list[dict],
        *,
        tools: list[dict] | None = None,
        response_format: Type[BaseModel] | None = None,
    ) -> str | BaseModel:
        """Single completion call. Returns str or a validated Pydantic object."""
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model,
                messages=messages,
                temperature=self.temperature,
                response_format=response_format,
            )
            self._track(resp.usage)
            return resp.choices[0].message.parsed

        kwargs: dict = dict(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
        )
        if tools:
            kwargs["tools"] = tools
        resp = self._client.chat.completions.create(**kwargs)
        self._track(resp.usage)
        return resp.choices[0].message.content

    def stream(self, messages: list[dict]) -> Generator[str, None, None]:
        """Yield completion tokens one chunk at a time."""
        for chunk in self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            stream=True,
        ):
            yield chunk.choices[0].delta.content or ""

    def run_tool_loop(
        self,
        messages: list[dict],
        tools: list[dict],
        tool_registry: dict[str, Callable],
        max_turns: int = 5,
    ) -> str:
        """Execute the tool-use loop until the model produces a final response."""
        messages = list(messages)
        for _ in range(max_turns):
            resp = self._client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=tools,
                temperature=self.temperature,
            )
            self._track(resp.usage)
            msg = resp.choices[0].message
            if resp.choices[0].finish_reason != "tool_calls":
                return msg.content
            messages.append(msg)
            for tc in msg.tool_calls:
                fn = tool_registry[tc.function.name]
                args = json.loads(tc.function.arguments)
                result = fn(**args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result),
                })
        return "[max_turns exceeded]"

    # ------------------------------------------------------------------
    # Cost tracking
    # ------------------------------------------------------------------

    @property
    def total_cost(self) -> float:
        """Cumulative USD cost for all calls made through this client."""
        if self.model not in PRICES:
            return 0.0
        p = PRICES[self.model]
        return (
            self._total_input_tokens * p["input"]
            + self._total_output_tokens * p["output"]
        ) / 1_000_000

    def reset_cost(self) -> None:
        self._total_input_tokens = 0
        self._total_output_tokens = 0

    def _track(self, usage) -> None:
        if usage:
            self._total_input_tokens += usage.prompt_tokens
            self._total_output_tokens += usage.completion_tokens

We verify all three modes — plain text, structured output, and cost tracking:

In [ ]:
llm = LLMClient()

# Test 1: plain text completion
result = llm.complete([{"role": "user", "content": "Say: Hello, Principal Engineer."}])
print(result)

# Test 2: structured output
entity = llm.complete(
    messages=[
        {"role": "system", "content": "Extract the financial entity."},
        {"role": "user", "content": "Goldman Sachs reported net revenue of $12.7B in Q3 2024."},
    ],
    response_format=FinancialEntity,
)
print(entity.model_dump_json(indent=2))

# Test 3: cost tracking
print(f"\nTotal cost: ${llm.total_cost:.6f}")
llm.reset_cost()
print(f"After reset: ${llm.total_cost:.6f}")

We use `llm` throughout the series. Every call accumulates cost, so we can measure the true API spend of any pipeline we build.

:::{.callout-note}
Save the `LLMClient` class to a file (e.g. `llm_client.py`) if you want to import it across notebooks without redefining it. In this series each notebook includes it inline for self-containment.

:::

---

$\blacksquare$